This notebook is used to build the vector stores for the CMPE 259 Project. Scrapped documents are located in a seperate notebook. (NOTE) This is a version of the notebook using local storage instead of MongoDB. (One VC in this notebook contains 1.4k chunks so it might take a bit to finish running) (RUN ONLY WHEN ABSOLUTELY NESSCARY)

# Setup

In [4]:
!pip install langchain_text_splitters langchain-community unstructured[all-docs]
!pip install langchain-huggingface
!pip install faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.1/476.1 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.7 MB/s eta 0:00:00
   ━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.6 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# Change project path [Anyone who is running this change to project folder]
project_path = '/content/drive/MyDrive/Colab Notebooks/CMPE259FinalProject'

%cd {project_path}

/content/drive/MyDrive/Colab Notebooks/CMPE259FinalProject


# Add Documents

In [5]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_core.documents import Document
import re
import os
import json

## Add Graduate Requirements + GAPE

In [6]:
PDF1 = "Documents/GradInfo/College of Graduate Studies - San José State University.pdf"
PDF2 = "Documents/GradInfo/Doctoral Requirements - San José State University.pdf"
PDF3 = "Documents/GradInfo/Graduate Admissions - San José State University.pdf"
PDF4 = "Documents/GradInfo/Master’s Requirements - San José State University.pdf"

In [7]:
# Build text_splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,  # chunk size (characters)
    chunk_overlap=300,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
# Load all PDFs into UnstructuredPDFLoader
loader1 = UnstructuredPDFLoader(PDF1)
loader2 = UnstructuredPDFLoader(PDF2)
loader3 = UnstructuredPDFLoader(PDF3)
loader4 = UnstructuredPDFLoader(PDF4)

# Add metadata to each document
doc1 = loader1.load_and_split()
doc2 = loader2.load_and_split()
doc3 = loader3.load_and_split()
doc4 = loader4.load_and_split()

def add_metadata(docs):
    for idx, doc in enumerate(docs):
        #Add filename
        doc.metadata['source'] = doc.metadata['source'].split('/')[-1]
    return docs

doc1 = add_metadata(doc1)
doc2 = add_metadata(doc2)
doc3 = add_metadata(doc3)
doc4 = add_metadata(doc4)


### GAPE Info

In [10]:
GAPE_INFO = 'CMPE259FinalContent/gape_info.json'

In [11]:
#Grab GAPE_INFO
with open(GAPE_INFO, 'r') as f:
    gape_info = json.load(f)



In [ ]:
GAPE_Docs = []

for item in gape_info:
    original_text = item['text']
    original_link = item['link']
    content_to_split = item['content']

    # Split the content into chunks using the predefined text_splitter
    chunks = text_splitter.split_text(content_to_split)

    for chunk in chunks:
        # Create a new Document object for each chunk
        new_doc = Document(
            page_content=chunk,
            metadata={'text': original_text, 'link': original_link}
        )
        GAPE_Docs.append(new_doc)

print(f"Number of GAPE_Docs created: {len(GAPE_Docs)}")

Number of GAPE_Docs created: 47


In [ ]:
# merge docs
GradReqDocs = doc1 + doc2 + doc3 + doc4 + GAPE_Docs
print(f"Number of documents: {len(GradReqDocs)}")

Number of documents: 69


## Add Graduate Major Information

In [12]:
MajorReqsFolder = "CMPE259FinalContent/GradMajorReqsDocs"

#Open Folder and print count
files = os.listdir(MajorReqsFolder)
print(f"Number of files: {len(files)}")


Number of files: 101


In [ ]:
headers = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers)

chunk_size = 1500
chunk_overlap = 100
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

def chunk_docs(markdown_content):
  return text_splitter.split_documents(markdown_splitter.split_text(markdown_content))



In [ ]:
def process_into_tag(filename):
  filename = filename.replace(".md", "")
  filename = filename.replace("[Masters]", "mstr")
  filename = filename.replace("[Doctoral]", "dr")
  filename = filename.replace(",", "")
  filename = filename.replace("-", "")
  filename = re.sub(r'\([^)]*\)', '', filename)
  filename = filename.replace("and", "")
  filename = re.sub(r'\bof\b', '', filename, flags=re.IGNORECASE)
  filename = re.sub(r'\bto\b', '', filename, flags=re.IGNORECASE)
  filename = re.sub(r'\bthe\b', '', filename, flags=re.IGNORECASE)
  filename = filename.replace("Engineering", "engr")
  filename = filename.replace("Technology", "tech")
  filename = filename.replace("technology", "tech")
  filename = filename.replace("Psychology", "psych")
  filename = filename.replace("Economics", "econ")
  filename = filename.replace("Science", "sci")
  filename = filename.replace("Sciences", "sci")
  filename = filename.replace("Biological", "bio")
  filename = filename.replace("Biology", "bio")
  filename = filename.replace("Educational", "edu")
  filename = filename.replace("Education", "edu")
  filename = filename.replace("Concentration", "concn")
  filename = filename.replace("Leadership", "ldr")
  filename = filename.lower()
  filename = filename.replace(" ", "_")
  filename = filename = filename.replace("industrial", "ind")
  return filename

In [ ]:
def extract_major(filename):
  filename = filename.replace(".md", "")
  filename = filename.replace("[Masters]", "")
  filename = filename.replace("[Doctoral]", "")
  return filename

In [ ]:
def load_markdown_files(folder_path):
    count = 0
    documents = []
    tags = {}
    for filename in os.listdir(folder_path):
        if filename.endswith(".md"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as file:
                markdown_content = file.read()
                chunked_content = chunk_docs(markdown_content)
                # save content to documents
                major = process_into_tag(filename)
                tags[major] = extract_major(filename)
                count += 1
                for chunk in chunked_content:
                  chunk.metadata['major'] = major
                  documents.append(chunk)
    print("# of Files Processed:",count)
    return documents,tags

MajorReqsDocs = load_markdown_files(MajorReqsFolder)

# of Files Processed: 101


In [ ]:
# [0] = All Documents
# [1] = Major Tag
print(len(MajorReqsDocs[0]),MajorReqsDocs[1])

1366 {'mstr_accounting__analytics_ms': ' Accounting and Analytics, MS', 'mstr_aerospace_engr_ms': ' Aerospace Engineering, MS', 'mstr_applied_anthropology_ma': ' Applied Anthropology, MA', 'mstr_applied_data_intelligence_ms': ' Applied Data Intelligence, MS', 'mstr_applied_mathematics_ms': ' Applied Mathematics, MS', 'mstr_archives__records_administration_mara': ' Archives and Records Administration, MARA', 'mstr_art_art_history__visual_culture_concn_ma': ' Art, Art History and Visual Culture Concentration, MA', 'mstr_art_digital_media_art_concn_mfa': ' Art, Digital Media Art Concentration, MFA', 'mstr_art_photography_concn_mfa': ' Art, Photography Concentration, MFA', 'mstr_art_pictorial_art_concn_mfa': ' Art, Pictorial Art Concentration, MFA', 'mstr_art_spatial_art_concn_mfa': ' Art, Spatial Art Concentration, MFA', 'mstr_artificial_intelligence_ms': ' Artificial Intelligence, MS', 'mstr_bioinformatics_ms': ' Bioinformatics, MS', 'mstr_bio_scis_ecology__evolution_concn_ms': ' Biologi

In [ ]:
MajorReqsDocs[0][1]

Document(metadata={'Header 1': 'Accounting and Analytics, MS', 'Header 2': 'Purpose of the MSAA Program', 'major': 'mstr_accounting__analytics_ms'}, page_content='The MSAA program prepares students for a career in public or corporate accounting by providing them with a strong foundation for both public accounting certification (CPA) and career advancement.</br>')

In [ ]:
# dump major tags
with open('CMPE259FinalContent/major_tags.json', 'w') as f:
    json.dump(MajorReqsDocs[1], f)
    print("Major Tags Dumped")

Major Tags Dumped


## Add Registrar Info

In [ ]:
Registrar_INFO = 'CMPE259FinalContent/registrar_data.json'

#Grab GAPE_INFO
with open(Registrar_INFO, 'r') as f:
    reg_info = json.load(f)


In [ ]:
reg_Docs = []

for item in reg_info:
    original_link = item['link']
    content_to_split = item['content']

    # Split the content into chunks using the predefined text_splitter
    chunks = text_splitter.split_text(content_to_split)

    for chunk in chunks:
        # Create a new Document object for each chunk
        new_doc = Document(
            page_content=chunk,
            metadata={'link': original_link}
        )
        reg_Docs.append(new_doc)

print(f"Number of reg_Docs created: {len(reg_Docs)}")

Number of reg_Docs created: 19


# Build Vector Stores

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/multi-qa-mpnet-base-cos-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
GradInfoVC = FAISS.from_documents(GradReqDocs, embeddings,distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

In [ ]:
GradMajorReqsVC = FAISS.from_documents(MajorReqsDocs[0], embeddings,distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

In [ ]:
RegVC = FAISS.from_documents(reg_Docs, embeddings,distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

In [ ]:
INDEX_DIR = "CMPE259FinalContent/FAISS"
os.makedirs(INDEX_DIR, exist_ok=True)

# Save
GradInfoVC.save_local(INDEX_DIR+"/GradInfo")
print("Saved GradInfoVC to:", INDEX_DIR+"/GradInfo")
GradMajorReqsVC.save_local(INDEX_DIR+"/GradMajorReqs")
print("Saved GradMajorReqsVC to:", INDEX_DIR+"/GradMajorReqs")
RegVC.save_local(INDEX_DIR+"/Reg")
print("Saved RegVC to:", INDEX_DIR+"/Reg")



Saved GradInfoVC to: /content/drive/MyDrive/CMPE259FinalContent/FAISS/GradInfo
Saved GradMajorReqsVC to: /content/drive/MyDrive/CMPE259FinalContent/FAISS/GradMajorReqs
Saved RegVC to: /content/drive/MyDrive/CMPE259FinalContent/FAISS/Reg
